# Level 1: Build an image AI from scratch

<a href="https://colab.research.google.com/github/kolk42254-code/ai-ml-goal/blob/main/Projects_by_AI/image-ai/level1_diffusion_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

You will build a **diffusion model**: the core idea behind Stable Diffusion. It studies thousands of real images, then draws **new** images that never existed.

**What "from scratch" means here**
- You write the noise maths and the neural network yourself, in PyTorch. No ready-made image model.
- Hugging Face gives you the training images (the `datasets` library) and a free place to share the finished model (the Hub).

**Cost: $0.** Everything runs on Google Colab's free GPU (usually a T4; Google doesn't guarantee which GPU you get).

**Time:** training takes roughly 10 minutes on a T4. That is my estimate: I tested this notebook on a CPU, not on a T4. Reading and running the rest takes about 30 minutes.

| Part | What happens |
|---|---|
| 0 | Setup: GPU check and installs |
| 1 | Settings: every knob in one place |
| 2 | Load real images from Hugging Face |
| 3 | Destroy images with noise (the *forward process*) |
| 4 | Build the neural network (a *U-Net*) |
| 5 | Train it |
| 6 | Generate new images |
| 7 | (Optional) Check your maths against Hugging Face `diffusers` |
| 8 | (Optional) Share your model on the Hugging Face Hub |
| 9 | Experiments, interview questions, and what's next |

**How to use this notebook**
1. Run the cells in order with **Shift + Enter**.
2. When a cell says **Predict first**, write your guess on paper *before* you run it, then compare.
3. Read the comments in the code: every line says what it does.

## Part 0: Setup

**In Colab:** open **Runtime → Change runtime type** and check that **T4 GPU** is selected. If it isn't, select it and click **Save**.

The next cell installs the Hugging Face libraries. Colab already has most of them, so it's quick.

In [ ]:
%pip install -q datasets huggingface_hub safetensors

In [ ]:
import copy
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import PyTorchModelHubMixin
from IPython.display import Image as ShowImageFile, display
from PIL import Image, ImageOps

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch version:", torch.__version__)
print("Device:", torch.cuda.get_device_name(0) if device == "cuda" else "CPU (no GPU found)")
if device == "cpu":
    print("Training will be much slower. In Colab: Runtime > Change runtime type > T4 GPU.")
    print("On a laptop with no GPU, use the smaller CPU settings in Part 1.")

## Part 1: Settings

Every knob lives here. To try something new, change it here and run the notebook again from this cell.

In [ ]:
# ---- Which images to learn from ----
DATASET = "mnist"   # "mnist" = handwritten digits, "fashion" = clothes, "butterflies" = colour photos

DATASETS = {
    # name:          Hugging Face dataset ID                           colour channels   mirror images?
    "mnist":       {"hf_id": "ylecun/mnist",                           "channels": 1, "flip": False},
    "fashion":     {"hf_id": "zalando-datasets/fashion_mnist",         "channels": 1, "flip": False},
    "butterflies": {"hf_id": "huggan/smithsonian_butterflies_subset",  "channels": 3, "flip": True},
}
# "flip" mirrors random training images left-right: a mirrored butterfly is still a butterfly,
# but a mirrored 7 is not a 7.

# ---- Model and training ----
IMAGE_SIZE = 32       # every image becomes 32 x 32 pixels
BASE_CHANNELS = 64    # width of the network: bigger = can learn more, but slower
TRAIN_STEPS = 6000    # number of training updates: more = better images, slower
BATCH_SIZE = 128      # images per update
LEARNING_RATE = 1e-3  # size of each weight change
T = 1000              # number of noise steps
SEED = 42             # fixes the random numbers, so runs repeat (almost exactly, on a GPU)

# ---- No GPU? Remove the # from these 3 lines for a smaller, faster run ----
# (On a 4-core cloud CPU this trained in about 10 minutes and drew readable but rough digits.
#  Generating is slow on a CPU too: expect several more minutes in Part 6.)
# BASE_CHANNELS = 32
# TRAIN_STEPS = 2500
# BATCH_SIZE = 64

torch.manual_seed(SEED)
cfg = DATASETS[DATASET]
print(DATASET, cfg)

## Part 2: Load the images from Hugging Face

`load_dataset()` downloads a dataset from the Hugging Face Hub the first time, then reuses the saved copy.

Then every picture becomes numbers:
- Each image becomes a **tensor** of shape `(channels, 32, 32)`. A tensor is PyTorch's version of a NumPy array; it can live on the GPU.
- Pixel values go from `0..255` to **`-1..1`**. Why? The noise we will add is centred on 0, so the images should be centred on 0 too.
- All images are stacked into one big tensor, `data`, of shape `(number_of_images, channels, 32, 32)`, and moved to the GPU once. Training then never waits for the disk.

In [ ]:
dataset = load_dataset(cfg["hf_id"], split="train")
print(dataset)


def image_to_tensor(pil_image, channels, size):
    """PIL image -> tensor of shape (channels, size, size) with values from -1 to 1."""
    mode = "L" if channels == 1 else "RGB"           # L = greyscale, RGB = colour
    img = pil_image.convert(mode)
    # Shrink to fit a size x size square. Non-square photos get white borders
    # (the butterfly photos are wide and already have white backgrounds).
    img = ImageOps.pad(img, (size, size), method=Image.BICUBIC, color="white")
    pixels = np.asarray(img, dtype=np.float32) / 255  # 0..255 -> 0..1
    if channels == 1:
        pixels = pixels[None, :, :]                  # (H, W)    -> (1, H, W)
    else:
        pixels = pixels.transpose(2, 0, 1)           # (H, W, 3) -> (3, H, W): PyTorch wants channels first
    return torch.from_numpy(pixels) * 2 - 1          # 0..1 -> -1..1


start = time.time()
data = torch.stack([image_to_tensor(img, cfg["channels"], IMAGE_SIZE) for img in dataset["image"]])
data = data.to(device)
print(f"data: shape {tuple(data.shape)}, values {data.min().item():.0f} to {data.max().item():.0f}, "
      f"took {time.time() - start:.0f}s")

Two helpers for looking at images. `make_grid` tiles many small images into one big picture; `show_images` plots it.

In [ ]:
def make_grid(images, ncols=8, pad=2):
    """Tile images (N, C, H, W) with values -1..1 into one picture (values 0..1) for plotting."""
    images = (images.detach().cpu().clamp(-1, 1) + 1) / 2   # -1..1 -> 0..1
    n, c, h, w = images.shape
    nrows = math.ceil(n / ncols)
    grid = torch.ones(c, nrows * (h + pad) + pad, ncols * (w + pad) + pad)   # white background
    for i in range(n):
        row, col = divmod(i, ncols)
        top, left = pad + row * (h + pad), pad + col * (w + pad)
        grid[:, top:top + h, left:left + w] = images[i]
    return grid.permute(1, 2, 0).squeeze(-1).numpy()   # (C, H, W) -> (H, W, C); greyscale -> (H, W)


def show_images(images, title="", ncols=8, save_as=None):
    """Plot a batch of images in a grid. If save_as is a file name, also save the grid as a PNG."""
    grid = make_grid(images, ncols)
    plt.figure(figsize=(ncols * 1.1, math.ceil(len(images) / ncols) * 1.1 + 0.4))
    plt.imshow(grid, cmap="gray", vmin=0, vmax=1)
    plt.title(title)
    plt.axis("off")
    plt.show()
    if save_as:
        plt.imsave(save_as, grid, cmap="gray", vmin=0, vmax=1)


show_images(data[:32], f"32 real training images ({DATASET})")

## Part 3: Destroy images with noise (the forward process)

**The big idea, in 3 steps**
1. **Destroy:** take a real image and add a little random noise, again and again, for `T = 1000` steps. At the end it is pure noise, like TV static.
2. **Learn to repair:** train a neural network to look at a noisy image and guess *which noise* was added.
3. **Create:** start from brand-new random noise and remove the guessed noise, step by step, 1000 times. A new image appears.

**The noise schedule** (three lists of 1000 numbers, one per step)
- `betas[t]`: how much fresh noise step `t` adds. It grows slowly from 0.0001 to 0.02.
- `alphas[t] = 1 - betas[t]`: how much of the image survives step `t`.
- `alpha_bars[t] = alphas[0] × alphas[1] × ... × alphas[t]`: how much of the *original* image is left after step `t`. Near 1 = almost clean. Near 0 = almost pure noise.

**The shortcut formula.** To get the image at step 500 you don't need to add noise 500 times. One line jumps straight there:

$$x_t = \sqrt{\bar\alpha_t}\; x_0 + \sqrt{1 - \bar\alpha_t}\; \varepsilon$$

- $x_0$ is the clean image, $\varepsilon$ (epsilon) is random noise from a normal distribution (mean 0, standard deviation 1), and $\bar\alpha_t$ is `alpha_bars[t]`.
- This shortcut is why training is fast: each training image can jump straight to any random step.

**Predict first:** write down your guesses for `alpha_bars[0]` (the start), `alpha_bars[499]` (halfway) and `alpha_bars[999]` (the end). Near 1, near 0.5, or near 0?

In [ ]:
betas = torch.linspace(1e-4, 0.02, T, device=device)   # fresh noise per step: 0.0001 ... 0.02
alphas = 1 - betas                                      # image kept per step
alpha_bars = torch.cumprod(alphas, dim=0)               # image left after step t (a running product)

for t in [0, T // 2 - 1, T - 1]:
    print(f"alpha_bars[{t}] = {alpha_bars[t].item():.5f}")

Halfway through, the image is already mostly noise, because the `alphas` *multiply*: every step keeps a little less of what was left.

Now the shortcut formula as a function, and a look at it in action.

In [ ]:
def add_noise(x0, t, noise):
    """Jump from clean images x0 straight to step t (one step number per image)."""
    a = alpha_bars[t].view(-1, 1, 1, 1)   # shape (N, 1, 1, 1), so it broadcasts over (C, H, W)
    return a.sqrt() * x0 + (1 - a).sqrt() * noise


steps_to_show = [0, T // 20, T // 10, T // 5, 3 * T // 10, 2 * T // 5, 3 * T // 5, T - 1]
k = len(steps_to_show)
x0 = data[:6].repeat_interleave(k, dim=0)                    # each image k times: a a a ... b b b ...
t = torch.tensor(steps_to_show, device=device).repeat(6)    # the steps, once per image: 0 50 ... 999 0 50 ...
noisy = add_noise(x0, t, torch.randn_like(x0))
show_images(noisy, f"Each row: one image after steps {steps_to_show}", ncols=k)

## Part 4: Build the neural network (a U-Net)

**Its job:** input = a noisy image + its step number `t`. Output = a guess of the noise, with exactly the same shape as the image.

**Why a U-Net** (it is shaped like the letter U):
- **Down path:** the image gets smaller (32 → 16 → 8 pixels) while the number of channels grows. Small and deep means it sees the big picture ("this is a 7").
- **Up path:** the image grows back to 32 × 32 to draw the fine details.
- **Skip connections:** the down path passes copies of its outputs straight across to the up path, so details lost while shrinking come back.

```
noisy image 32x32
     |
   first
     |
   down1  32x32 -------------------------------- copy -------------------+
     | shrink                                                            |
   down2  16x16 --------------- copy ---------------+                    |
     | shrink                                       |                    |
   down3   8x8 --- copy ---+                        |                    |
     |                     v                        v                    v
   middle  8x8 ------->  up3  8x8 --- grow --->  up2 16x16 --- grow --->  up1 32x32 ---> last ---> noise guess
```

**The building blocks** (each one is a class):
- `TimeEmbedding` turns the step number `t` (like 37 or 912) into a list of numbers made from sine and cosine waves of different speeds. Like the hands of a clock (seconds, minutes, hours), the waves together pin down exactly which step it is. Transformers use the same trick for word positions.
- `ResBlock`: two convolutions, plus the time signal, plus a **residual connection**: output = input + change. Deep networks train much more easily this way.
- `TinyUNet` wires the blocks into a U.

**Words you'll see:**
- **Convolution** (`nn.Conv2d`): a small 3 × 3 filter that slides over the image looking for patterns (edges, curves, loops).
- **Channels:** how many different patterns are stored per pixel. A greyscale image has 1, a colour image 3; inside the network we use 64 to 128.
- **GroupNorm:** rescales numbers inside the network so they stay in a healthy range.
- **SiLU:** the activation function, the non-linear "bend" that lets a network learn curves, not only straight lines.

**Classes, in action** (your current topic, P66-P71, and the next one, P73-P74 inheritance and `super()`): every block is a class that **inherits** from `nn.Module` and calls `super().__init__()`. `TinyUNet` has two parents: `nn.Module` makes it a network, and `PyTorchModelHubMixin` (from Hugging Face) gives it `save_pretrained()`, `push_to_hub()` and `from_pretrained()` for free.

In [ ]:
class TimeEmbedding(nn.Module):
    """Step numbers t, shape (N,) -> vectors, shape (N, dim * 4)."""

    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.SiLU(), nn.Linear(dim * 4, dim * 4))

    def forward(self, t):
        half = self.dim // 2
        speeds = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)   # 1 ... about 1/10000
        angles = t.float()[:, None] * speeds[None, :]                                      # (N, half)
        waves = torch.cat([angles.sin(), angles.cos()], dim=1)                             # (N, dim)
        return self.mlp(waves)


class ResBlock(nn.Module):
    """Two 3x3 convolutions + the time signal + a residual (shortcut) connection."""

    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_channels)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        # The shortcut needs a 1x1 convolution only when the number of channels changes.
        if in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.shortcut = nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_proj(t_emb)[:, :, None, None]   # add the time signal to every pixel
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.shortcut(x)                       # output = input + change


class TinyUNet(nn.Module, PyTorchModelHubMixin):
    """Noisy images (N, C, 32, 32) + steps (N,) -> noise guesses (N, C, 32, 32)."""

    def __init__(self, image_channels=1, base=64):
        super().__init__()
        c1, c2 = base, base * 2   # channels at 32x32, and at 16x16 and 8x8
        time_dim = base * 4
        self.time_embedding = TimeEmbedding(base)
        self.first = nn.Conv2d(image_channels, c1, kernel_size=3, padding=1)
        # Down path
        self.down1 = ResBlock(c1, c1, time_dim)                            # 32x32
        self.shrink1 = nn.Conv2d(c1, c1, kernel_size=3, stride=2, padding=1)   # 32 -> 16
        self.down2 = ResBlock(c1, c2, time_dim)                            # 16x16
        self.shrink2 = nn.Conv2d(c2, c2, kernel_size=3, stride=2, padding=1)   # 16 -> 8
        self.down3 = ResBlock(c2, c2, time_dim)                            # 8x8
        # Bottom of the U
        self.middle = ResBlock(c2, c2, time_dim)                           # 8x8
        # Up path: each block also receives the copy from the down path, so its input channels add up
        self.up3 = ResBlock(c2 + c2, c2, time_dim)                         # 8x8
        self.up2 = ResBlock(c2 + c2, c1, time_dim)                         # 16x16
        self.up1 = ResBlock(c1 + c1, c1, time_dim)                         # 32x32
        self.grow = nn.Upsample(scale_factor=2, mode="nearest")            # 8 -> 16 -> 32
        self.last = nn.Sequential(nn.GroupNorm(8, c1), nn.SiLU(),
                                  nn.Conv2d(c1, image_channels, kernel_size=3, padding=1))

    def forward(self, x, t):
        t_emb = self.time_embedding(t)
        x = self.first(x)
        d1 = self.down1(x, t_emb)                                   # (N, c1, 32, 32)
        d2 = self.down2(self.shrink1(d1), t_emb)                    # (N, c2, 16, 16)
        d3 = self.down3(self.shrink2(d2), t_emb)                    # (N, c2, 8, 8)
        h = self.middle(d3, t_emb)                                  # (N, c2, 8, 8)
        h = self.up3(torch.cat([h, d3], dim=1), t_emb)              # copy from down3 -> (N, c2, 8, 8)
        h = self.up2(torch.cat([self.grow(h), d2], dim=1), t_emb)   # copy from down2 -> (N, c1, 16, 16)
        h = self.up1(torch.cat([self.grow(h), d1], dim=1), t_emb)   # copy from down1 -> (N, c1, 32, 32)
        return self.last(h)

**Predict first:** we feed in 8 noisy images of shape `(8, channels, 32, 32)`. What shape comes out?

In [ ]:
model = TinyUNet(image_channels=cfg["channels"], base=BASE_CHANNELS).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"TinyUNet has {n_params:,} weights")

x = torch.randn(8, cfg["channels"], IMAGE_SIZE, IMAGE_SIZE, device=device)
t = torch.randint(0, T, (8,), device=device)
print("input shape: ", tuple(x.shape))
print("output shape:", tuple(model(x, t).shape))

Same shape in and out: for every pixel, the network guesses the noise in that pixel. For scale: the U-Net inside Stable Diffusion v1 has about 860 million weights.

## Part 5: Train the network

**One training step** (this is all of DDPM training: Algorithm 1 in [Ho et al., 2020](https://arxiv.org/abs/2006.11239)):
1. Take a random batch of real images `x0`.
2. Pick a random step `t` for each image.
3. Make random noise and jump to step `t`: `x_t = add_noise(x0, t, noise)`.
4. Ask the network for its noise guess: `model(x_t, t)`.
5. Score it: **loss** = mean squared error between the guess and the real noise. Backpropagation works out how each weight should change; the optimizer (AdamW) changes them.

**EMA (exponential moving average):** we also keep a second copy of the network whose weights move slowly towards the training weights. It averages out the jitter of single training steps, so its images are cleaner. Think of your average score over your last 1000 practice runs, instead of your last run only.

**Predict first:** a lazy network that always answers "no noise" (all zeros) gets what loss? Hint: the noise has mean 0 and standard deviation 1, and MSE = the average of (guess − noise)². Write your answer, then watch the first printed loss. Your answer is the **baseline**: a loss far below it means the network really learned something.

In [ ]:
ema_model = copy.deepcopy(model).eval()   # the slow-moving average copy: we draw images with it
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


def update_ema(ema_model, model, step, max_decay=0.999):
    """Move each EMA weight a tiny bit towards the training weight: ema = decay * ema + (1 - decay) * w."""
    decay = min(max_decay, (1 + step) / (10 + step))   # warm-up: shorter memory at the start (same rule as diffusers' EMAModel)
    with torch.no_grad():
        for ema_w, w in zip(ema_model.parameters(), model.parameters()):
            ema_w.copy_(decay * ema_w + (1 - decay) * w)

In [ ]:
losses = []
start = time.time()
model.train()
for step in range(1, TRAIN_STEPS + 1):
    # 1. A random batch of real images
    idx = torch.randint(0, len(data), (BATCH_SIZE,), device=device)
    x0 = data[idx]
    if cfg["flip"]:   # mirror about half of them: more variety for free
        mirror = torch.rand(BATCH_SIZE, device=device) < 0.5
        x0 = torch.where(mirror[:, None, None, None], x0.flip(dims=[3]), x0)

    # 2. A random step for each image, and 3. jump there with the shortcut formula
    t = torch.randint(0, T, (BATCH_SIZE,), device=device)
    noise = torch.randn_like(x0)
    x_t = add_noise(x0, t, noise)

    # 4. The network guesses the noise, and 5. we score the guess
    predicted_noise = model(x_t, t)
    loss = F.mse_loss(predicted_noise, noise)

    # Learn: compute the gradients, cap their size so one bad batch can't wreck the weights, update
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    update_ema(ema_model, model, step)

    losses.append(loss.item())
    if step == 1 or step % 500 == 0:
        print(f"step {step:5d}/{TRAIN_STEPS} | loss {np.mean(losses[-500:]):.4f} | {time.time() - start:5.0f}s")

print(f"Training done in {(time.time() - start) / 60:.1f} minutes")

In [ ]:
window = min(100, len(losses))
smooth = np.convolve(losses, np.ones(window) / window, mode="valid")   # moving average, to see the trend
plt.figure(figsize=(8, 3))
plt.plot(losses, alpha=0.3, label="each step")
plt.plot(range(window - 1, len(losses)), smooth, label=f"average of {window} steps")
plt.axhline(1.0, color="red", linestyle="--", label='baseline: always guess "no noise"')
plt.yscale("log")
plt.xlabel("training step")
plt.ylabel("loss (MSE)")
plt.legend()
plt.show()

ema_model.save_pretrained(f"tiny-diffusion-{DATASET}")   # config.json + model.safetensors
print(f"Saved to the folder tiny-diffusion-{DATASET}. In Colab, files vanish when the runtime stops: "
      "download them, or share the model in Part 8.")

## Part 6: Generate new images

**Generating** (Algorithm 2 in the same paper):
- Start with pure noise `x`.
- For `t = 999, 998, ..., 0`:
  - the network guesses the noise: `predicted_noise = model(x, t)`;
  - remove a little of it: `mean = (x - betas[t] / sqrt(1 - alpha_bars[t]) * predicted_noise) / sqrt(alphas[t])`;
  - add a small random kick: `x = mean + sqrt(betas[t]) * new_noise` (skipped at the very last step).

**Why add new noise while removing noise?** Each step is a *probability distribution*, not one fixed answer: `mean` is its centre and the kick picks a point around it. Experiment 3 in Part 9 shows what happens without the kick.

**Predict first:** how many times does the network run to generate one batch of images?

In [ ]:
@torch.no_grad()   # no training here: skip the gradient bookkeeping (faster, less memory)
def generate(model, n, keep_every=None):
    """Start from pure noise and remove noise T times. Returns n images, plus snapshots if keep_every is set."""
    model.eval()
    x = torch.randn(n, cfg["channels"], IMAGE_SIZE, IMAGE_SIZE, device=device)   # pure noise
    snapshots = []
    for t in reversed(range(T)):   # 999, 998, ..., 0
        t_batch = torch.full((n,), t, device=device, dtype=torch.long)
        predicted_noise = model(x, t_batch)
        mean = (x - betas[t] / (1 - alpha_bars[t]).sqrt() * predicted_noise) / alphas[t].sqrt()
        if t > 0:
            x = mean + betas[t].sqrt() * torch.randn_like(x)   # small random kick
        else:
            x = mean                                            # last step: no kick
        if keep_every and t % keep_every == 0:
            snapshots.append(x.cpu())
    return x, snapshots


torch.manual_seed(SEED)
start = time.time()
samples, _ = generate(ema_model, 64)
print(f"Generated 64 images in {time.time() - start:.0f}s (the network ran {T} times)")
show_images(samples, f"64 NEW images drawn by your model ({DATASET})", save_as="samples.png")

**Watch it happen.** The rows below are 8 new images. Each column is a later moment in the generation, from pure noise (left) to the final image (right). The animation shows 50 moments (every 20th step when `T = 1000`).

In [ ]:
torch.manual_seed(SEED + 1)
final, snapshots = generate(ema_model, 8, keep_every=max(1, T // 50))

fractions = [0, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0]   # how far through generation each column is
picked = [snapshots[round(f * (len(snapshots) - 1))] for f in fractions]
film = torch.stack(picked, dim=1).flatten(0, 1)        # (8 images, 8 moments, C, H, W) -> (64, C, H, W)
show_images(film, "Each row: noise -> image (left to right: " + ", ".join(f"{f:.0%}" for f in fractions) + " done)",
            ncols=len(fractions))

# Animated GIF: one frame per snapshot
frames = []
for x in snapshots:
    pixels = (make_grid(x, ncols=8) * 255).astype(np.uint8)
    frame = Image.fromarray(pixels)
    frames.append(frame.resize((frame.width * 4, frame.height * 4), Image.NEAREST))   # 4x bigger, sharp pixels
durations = [80] * (len(frames) - 1) + [2000]   # milliseconds per frame; hold the final image for 2 s
frames[0].save("denoising.gif", save_all=True, append_images=frames[1:], duration=durations, loop=0)
display(ShowImageFile(filename="denoising.gif"))

## Part 7 (optional): Check your maths against Hugging Face `diffusers`

You wrote the noise schedule and `add_noise()` by hand. `diffusers` is Hugging Face's library for diffusion models, and its `DDPMScheduler` does the same job. If both give the same numbers, your maths matches the professional version.

In [ ]:
%pip install -q diffusers
from diffusers import DDPMScheduler

scheduler = DDPMScheduler(num_train_timesteps=T, beta_start=1e-4, beta_end=0.02, beta_schedule="linear")

x0 = data[:4].cpu()
noise = torch.randn_like(x0)
t = torch.tensor([0, T // 100, T // 2, T - 1])
ours = add_noise(x0.to(device), t.to(device), noise.to(device)).cpu()
theirs = scheduler.add_noise(x0, noise, t)

print("alpha_bars: largest difference =", (alpha_bars.cpu() - scheduler.alphas_cumprod).abs().max().item())
print("add_noise:  largest difference =", (ours - theirs).abs().max().item())
print("(0, or something tiny like 1e-7 from rounding on a GPU, means the maths is the same.)")

## Part 8 (optional): Share your model on the Hugging Face Hub

Free. Your model gets a public page that anyone can download from: a good link for your GitHub profile and resume.

1. Make a free account at [huggingface.co/join](https://huggingface.co/join).
2. Open **Settings → Access Tokens → Create new token**, choose the **Write** type, and copy the token.
3. Run the next cell and paste the token into the box. Never put a token in your code or on GitHub.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
username = api.whoami()["name"]
repo_id = f"{username}/tiny-diffusion-{DATASET}"

ema_model.push_to_hub(repo_id)   # creates the repo if needed, uploads config.json + model.safetensors
for file_name in ["samples.png", "denoising.gif"]:
    api.upload_file(path_or_fileobj=file_name, path_in_repo=file_name, repo_id=repo_id)

model_card = f"""---
tags:
- diffusion
- ddpm
- unconditional-image-generation
- pytorch_model_hub_mixin
---
# Tiny diffusion model ({DATASET})

A small DDPM ({n_params / 1e6:.1f} million weights) written from scratch in PyTorch, trained for
{TRAIN_STEPS} steps on [{cfg['hf_id']}](https://huggingface.co/datasets/{cfg['hf_id']}) at {IMAGE_SIZE}x{IMAGE_SIZE} pixels.

![64 generated images](samples.png)

![From noise to image](denoising.gif)
"""
api.upload_file(path_or_fileobj=model_card.encode(), path_in_repo="README.md", repo_id=repo_id)
print("Your model page:", f"https://huggingface.co/{repo_id}")

Now load it back the way anyone else in the world would, and let it draw:

In [ ]:
downloaded = TinyUNet.from_pretrained(repo_id).to(device)
torch.manual_seed(0)
images, _ = generate(downloaded, 16)
show_images(images, f"Drawn by the model downloaded from {repo_id}")

## Part 9: Experiments, questions, and what's next

### Experiments (predict first, then run)
Change one thing at a time in **Part 1** (or where the experiment says), then run again from there.

1. **Train less:** `TRAIN_STEPS = 1000`. How will the images change?
2. **Colour:** `DATASET = "butterflies"`. There are only 1000 photos. What could go wrong when a model studies so few images for so long?
3. **No kick:** in `generate()`, change `x = mean + betas[t].sqrt() * torch.randn_like(x)` to `x = mean`, then run Part 6 again (no retraining needed).
4. **Fewer noise steps:** `T = 200`, and retrain. Before training, check `alpha_bars[-1]`. Is the last step still pure noise? Generation always starts from pure noise, so what problem does that cause?

### Interview questions
Answer out loud or on paper first, then open the answers.

1. Why do we train on random steps `t` instead of going through all 1000 steps in order?
2. Why does the network need `t` as an input?
3. Why does the U-Net need skip connections?
4. Why is generating slow, when one training step is fast?
5. Why scale the pixels to `-1..1`?

<details>
<summary>Answers</summary>

1. The goal is a low error at *every* step. A random step per image gives each batch a fair mix of all the steps, and the shortcut formula makes any step one line to reach. Going in order would practise one step at a time, so the network would tune itself to whichever step came last and drift away from the others.
2. The same noisy-looking image needs different repairs at different steps: at step 900 the noise is huge, at step 10 it is tiny. `t` tells the network how much noise to expect.
3. Shrinking to 8 × 8 throws away fine detail. The copies carried across give the up path those details back.
4. Training uses the shortcut: one network call per image. Generating can't take a shortcut, because each step needs the result of the step before, so it takes `T` calls in a row. Faster samplers such as DDIM skip steps for this reason.
5. The noise is centred on 0, so images centred on 0 put the image and the noise on the same scale, and the shortcut formula keeps the total spread about the same at every step.

</details>

### Upgrades for later (Level 1.5)
- **"Draw me a 7":** give the network the digit label as a second input, next to `t`. This is called *class conditioning*.
- **Attention layers** in the U-Net, and a **cosine noise schedule**: both usually give better images.
- **DDIM sampling:** about 50 steps instead of 1000.

### Level 2
Fine-tune a real Stable Diffusion model on your own pictures with LoRA, using Hugging Face `diffusers`.